使用一个聪明的模型提取答案，按照提取LLM的格式输出
只运行单一的json
限制回答问题的LLM的输出，限定了给定格式

In [ ]:
import json
import os
from PIL import Image
import re
import ast
import string
import textwrap
import base64
import time
from openai import OpenAI
import io
import matplotlib.patches as mpatches

In [2]:
def generate_prompt(sample):
    items = sample["items"]
    caption_map = sample["caption"]
    question = sample["question"]
    answer_type = sample["answer_type"]

    # 输出格式规则
    if answer_type == "numeric":
        format_rule = "a single number"
    elif answer_type == "numeric_list":
        format_rule = "[a, b, c]"
    elif answer_type == "choice":
        format_rule = "A"
    elif answer_type == "choice_list":
        format_rule = "[A, B, C]"
    elif answer_type == "table_list":
        format_rule = "[[...], [...], ...]"
    elif answer_type == "code":
        format_rule = "```python\n<code>\n```"
    else:
        format_rule = "short text"

    # caption
    lines = []
    for item in items:
        caption_text = caption_map.get(item, "")
        lines.append(f"the caption of {item}: {caption_text}")
    caption = "\n".join(lines)

    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given the following items: {items}.
Each item has a corresponding caption:
{caption}

Question:
{question}

Instructions:
1. Analyze the provided items to answer the question.
2. Do NOT output reasoning, explanation, or intermediate steps.
3. Output ONLY the final answer.

Output format:
Answer: {format_rule}

Rules:
- Output exactly ONE line.
- The line must start with "Answer:".
- Do not output anything before or after the answer.
"""

    return prompt

In [3]:
# 转换图表格式 压缩图片-> 减少token
def encode_image_to_base64(image_path, max_size=768, quality=70):
    img = Image.open(image_path)

    # 等比例缩小
    img.thumbnail((max_size, max_size))
    
    if img.mode != "RGB":
        img = img.convert("RGB")

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality, optimize=True)

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [ ]:
# === 调用 LLM（自动重试直到成功返回content）===
def call_llm(subject_path, sample):
    items = sample["items"]
    
    # 生成提示词
    prompt = generate_prompt(sample)

    content_list = []
    path_map = sample["path"]

    for key in path_map:
        relative_path = path_map[key]
        full_path = os.path.join(subject_path, relative_path)

        if not os.path.exists(full_path):
            print(f"文件不存在: {full_path}")
            continue

        mime_type = "image/jpeg"
        image_base64 = encode_image_to_base64(full_path)

        content_list.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_base64}"
            }
        })

    # 添加文本
    content_list.append({
        "type": "text",
        "text": prompt
    })
    
    # 调用模型    
    client = OpenAI(
        base_url="",
        api_key="",
    )
    
    # 一直尝试直到调用成功
    attempt = 0
    base_sleep=1.0
    max_sleep=60.0
    MAX_RETRY = 50 # 避免死循环
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")
    
        try:
            resp = client.chat.completions.create(
                model="",
                messages=[{"role": "user", "content": content_list}],
                temperature=0,
            )
            return resp.choices[0].message.content
 
        except Exception as e:
            print(f"error: {e}")
            msg = str(e).lower()

            if ("401" in msg) or ("403" in msg) or ("400" in msg) or ("invalid" in msg) or ("not found" in msg):
                raise

            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1

In [ ]:
# 使用deepseek提取答案
def exact_answer_llm(output, answer_type):
    
    # 设计prompt
        # 输出规则 
    if answer_type == "numeric":
        output_format = "answer: {number}. Return exactly one number after 'answer:'. Do not include units, symbols, commas, explanations, or extra text."

    elif answer_type == "numeric_list":
        output_format = "answer: [a, b, c]. Return a list of numbers inside square brackets. No text inside brackets. No explanations."

    elif answer_type == "choice":
        output_format = "answer: {A capital letter}. Return exactly one capital letter after 'answer:'. No punctuation. No brackets. No extra text."

    elif answer_type == "choice_list":
        output_format = "answer: [A, B, C]. Return a list of capital letters inside square brackets. No explanations."

    elif answer_type == "table_list":
        output_format = "answer: [[row1_col1, row1_col2, ...], [row2_col1, row2_col2, ...], ...]. Return a nested list representing the table. Only structured list format. "

    elif answer_type == "code":
        output_format = "answer: ```python {code block}```. Return only executable code inside the code block. No explanation outside the code block."

    else:
        output_format = "answer: {text}. Return only the final short text answer after 'answer:'. The answer MUST end with a period '.'. Do not include anything after the period. No explanations or additional commentary."
        
    
    prompt = f"""
You are an answer extraction assistant. Your task is to extract the final answer from a model’s response according to the specified answer type. 
Answer type: {answer_type}
Model response:
{output}
You must extract only the final answer and strictly follow this output format:
{output_format}
Do not include explanations or any additional text.
Return exactly one answer in the required format.
    """
        
    # 调用llm 多次调用 直到成功返回 deepseek
    client = OpenAI(
        base_url="",
        api_key="",
    )
    
    attempt = 0
    base_sleep=1.0
    max_sleep=60.0
    MAX_RETRY = 50 # 避免死循环
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")
    
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": prompt}
            ],
                max_tokens=1024,
                temperature=0,
                stream=False
            )

            output_llm = response.choices[0].message.content
            
            if not output_llm:
                raise ValueError("empty response")
                
            return output_llm
 
        except Exception as e:
            msg = str(e).lower()

            if ("401" in msg) or ("403" in msg) or ("400" in msg) or ("invalid" in msg) or ("not found" in msg):
                raise

            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1

In [7]:
# 测评代码 —— numeric
# 提取数字 -> 计算相对误差 —> 相对误差小于tol则得1分

def evaluate_numeric(output, gt):
    eps_rel = 0.01
    eps_abs = 1e-6
    
    if output is None:
        return False

    output = output.strip()
    
    # 提取答案
    output_llm = exact_answer_llm(output, "numeric")
    
    number_pattern = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    # 提取answer:后面的数字
    match = re.search(rf"^answer:\s*({number_pattern})\s*$", output_llm.strip(), re.IGNORECASE)

    if match:
        pred = float(match.group(1))
    else:
        # 情况2：没有answer 则提取第一个数字
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", output_llm)
        
        if not match:
            return False  
        
        pred = float(match.group())

    gt = float(gt)

    tolerance = max(eps_abs, eps_rel * abs(gt))

    return abs(pred - gt) <= tolerance

In [8]:
# 测评代码 —— numeric_list
# 提取数字 -> 计算相对误差 —> 所有相对误差小于tol才算正确

def evaluate_numeric_list(output, gt):
    eps_rel = 0.01
    eps_abs = 1e-6

    if output is None:
        return False
    
    s = output.strip()
    
    output_llm = exact_answer_llm(s, "numeric_list")

    # 取answer后面的list
    number_pattern = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"
    match = re.search(rf"(?is)answer:\s*(\[\s*{number_pattern}(?:\s*,\s*{number_pattern})*\s*\])", output_llm)

    if match:
        list_str = match.group(1)
    else:
        # 提取第一个[...]
        match = re.search(rf"(\[\s*{number_pattern}(?:\s*,\s*{number_pattern})*\s*\])", output_llm)
        if match:
            list_str = match.group(1)
        else:
            return False

    # 解析成list并校验
    try:
        pred = ast.literal_eval(list_str)
    except Exception:
        return False

    if not isinstance(pred, list):
        return False

    if len(pred) != len(gt): # 元素个数不一致则回答错误
        return False

    for pi, gi in zip(pred, gt):
        try:
            pi = float(pi)
            gi = float(gi)
        except Exception:
            return False

        toli = max(eps_abs, eps_rel * abs(gi))
        if abs(pi - gi) > toli:
            return False

    return True
    

In [9]:
# 测评代码 —— text
# 衡量两个字符串的相似程度 用编辑距离衡量

# 规范化字符串
def normalize_text(s):
    if s is None:
        return ""
    
    s = str(s).lower().strip()
    
    # 去掉常见的中英文标点
    punct = string.punctuation + "，。！？；：“”‘’（）【】《》、…—·"
    s = s.translate(str.maketrans("", "", punct))
    
    # 空格合一
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 计算编辑距离
def edit_distance(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(
                prev[j] + 1,      # delete
                cur[j - 1] + 1,   # insert
                prev[j - 1] + cost  # substitute
            )
        prev = cur
    return prev[m]

# 测评
def evaluate_text(output, gt):
    threshold = 0.9
    
    if output is None:
        return False
    
    output_llm = exact_answer_llm(output, "text")
    
    # 提取answer: 后面的text 遇到.结束
    m = re.search(r"(?is)^\s*answer:\s*(.+?\.)", output_llm.strip())
    if not m:
        return False

    pred_text = m.group(1).strip()
    
    p = normalize_text(pred_text)
    g = normalize_text(gt)
    
    if len(p) == 0 and len(g) == 0:
        return True
    if len(p) == 0 or len(g) == 0:
        return False
        
    dist = edit_distance(p, g)
    sim = 1.0 - dist / max(len(p), len(g))
    
    return sim >= threshold
    
    

In [10]:
# 测评代码 —— choice
# 提取原则： 如果只有一个大写字母，则该字母为答案，否则提取左右两边都是标点的大写字母 若有多个 则用最后一个

def evaluate_choice(output, gt):
    s = str(output).strip()
    
    output_llm = exact_answer_llm(s, "choice")
    
    # 提取answer: 后面的第一个大小字母
    match = re.search(r"(?is)^\s*answer:\s*([A-Z])\s*(?:\r?\n|$)", output_llm.strip())
    if not match:
        return False

    pred = match.group(1)
    
    return pred == str(gt).strip()


In [11]:
# 测评代码 —— choice_list
def evaluate_choice_list(output, gt):
    if output is None:
        return False
    
    s = str(output).strip()
    
    output_llm = exact_answer_llm(s, "choice")
    
    # 提取 answer: 后面的第一个 [A, B, C]
    match = re.search(r"(?is)^\s*answer:\s*(\[\s*[A-Z](?:\s*,\s*[A-Z])*\s*\])", output_llm.strip())

    if not match:
        return False

    list_str = match.group(1)

    # 解析为Python list
    try:
        pred = ast.literal_eval(list_str)
    except Exception:
        return False

    if not isinstance(pred, list):
        return False

    # 去空格并标准化
    pred = [x.strip() for x in pred]
    
    if not all(re.fullmatch(r"[A-Z]", x) for x in pred):
        return False
    
    return set(pred) == set(gt)
    

In [12]:
# 测评代码 —— table_list

# 标准化行名和列名
def normalize_label(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 计算两个字符串的编辑距离
def _edit_distance(a, b):
    """
    Levenshtein edit distance (insert/delete/substitute).
    """
    a = normalize_label(a)
    b = normalize_label(b)
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n

    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(
                prev[j] + 1,        # delete
                cur[j - 1] + 1,     # insert
                prev[j - 1] + cost  # substitute
            )
        prev = cur
    return prev[m]

# 计算字符串相似度
def esm_sim(a, b):
    """
    ESM similarity:
    sim = 1 - ED(a,b) / max(len(a), len(b))
    """
    a_n = normalize_label(a)
    b_n = normalize_label(b)
    if len(a_n) == 0 and len(b_n) == 0:
        return 1.0
    if len(a_n) == 0 or len(b_n) == 0:
        return 0.0
    dist = _edit_distance(a_n, b_n)
    return 1.0 - dist / max(len(a_n), len(b_n))


# 计算数值相似度
def num_sim(p, g, eps=1e-12):
    """
    DePlot-style numeric distance and similarity:
      dist = min(1, |p-g| / max(|g|, eps))
      sim  = 1 - dist
    """
    try:
        if p is None or g is None:
            return 0.0
        p = float(p)
        g = float(g)
        denom = max(abs(g), eps)
        dist = abs(p - g) / denom
        dist = min(1.0, dist)
        return 1.0 - dist
    except Exception:
        return 0.0
    
    
# 将每个单元格整理成三元组 cells: list of (row_label, col_label, value)
def table_to_cells(table):

    # 1) 如果 table 是字符串（LLM输出常见），先转成 list
    if isinstance(table, str):
        try:
            table = ast.literal_eval(table.strip())
        except Exception:
            return []

    # 2) 基本结构检查
    if not isinstance(table, list) or len(table) < 2:
        return []

    if not all(isinstance(r, list) and len(r) >= 1 for r in table):
        return []

    # 3) 取列名（第一行除第一个）
    cols = table[0][1:]
    cells = []

    # 4) 遍历数据行
    for r in range(1, len(table)):
        row = table[r]
        if len(row) == 0:
            continue

        row_label = row[0]
        vals = row[1:]

        for j, col_label in enumerate(cols):
            val = vals[j] if j < len(vals) else None
            cells.append((row_label, col_label, val))

    return cells
    
# 匹配算法
# 简单贪心匹配
def greedy_max(weight):
    """
    最大化 sum(weight[i][j]) 的贪心一对一匹配
    返回 assignment: 长度 n，assignment[i] = j 或 -1
    """
    n = len(weight)
    m = len(weight[0]) if n else 0

    assignment = [-1] * n
    used_cols = set()

    # 收集所有可能匹配 (sim, i, j)
    pairs = []
    for i in range(n):
        for j in range(m):
            if weight[i][j] > 0:
                pairs.append((weight[i][j], i, j))

    # 按相似度降序排序
    pairs.sort(reverse=True)

    # 贪心匹配
    for sim, i, j in pairs:
        if assignment[i] == -1 and j not in used_cols:
            assignment[i] = j
            used_cols.add(j)

    return assignment


# 数字正则（你原来代码里用到的 _NUM）
_NUM = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def parse_table_list(output):    
    try:
        s = str(output).strip()
        
        output_llm = exact_answer_llm(s, "table_list")

        # 1) 先从 LLM 输出中提取 answer: 后面的“第一个二维 list” 
        m = re.search(r"(?is)^\s*answer:\s*(\[\s*\[.*?\]\s*(?:,\s*\[.*?\]\s*)*\])", output_llm)
        if m:
            table_str = m.group(1)
        else: # 没有answer:的情况
            m2 = re.search(r"(?is)(\[\s*\[.*?\]\s*(?:,\s*\[.*?\]\s*)*\])", output_llm)
            if not m2:
                return None
            
            table_str = m2.group(1)

        # 2) 把非数字的 label 自动加引号；数字/数字字符串保持数值
        pattern = r'(?<=\[|,)\s*([^\[\],]+?)\s*(?=,|\])'

        def fix_token(m):
            tok = m.group(1).strip()

            # 已带引号
            if (len(tok) >= 2) and (tok[0] == tok[-1]) and tok[0] in ("'", '"'):
                inner = tok[1:-1].strip()
                if _NUM.fullmatch(inner):
                    return inner
                inner_esc = inner.replace('"', '\\"')
                return f'"{inner_esc}"'

            # 不带引号：数字
            if _NUM.fullmatch(tok):
                return tok

            # 不带引号且非数字：视为字符串 label
            tok_esc = tok.replace('"', '\\"')
            return f'"{tok_esc}"'

        table_fixed = re.sub(pattern, fix_token, table_str)

        # 3) 解析成二维list格式
        table = ast.literal_eval(table_fixed)

        # 4) 基本结构校验：二维 list，且至少 2x2
        if not isinstance(table, list) or not all(isinstance(row, list) for row in table):
            return None
        if len(table) < 2 or len(table[0]) < 2:
            return None

        # 5) 类型规范化：第一行全转 str；第一列全转 str；其余尽量 float（缺失补 None）
        norm = []
        header = [str(x) for x in table[0]]
        norm.append(header)

        cols_n = len(header)

        for r in range(1, len(table)):
            row = table[r]
            if not isinstance(row, list) or len(row) == 0:
                return None

            row_label = str(row[0])
            new_row = [row_label]

            for j in range(1, cols_n):
                v = row[j] if j < len(row) else None
                if v is None:
                    new_row.append(None)
                    continue
                if isinstance(v, (int, float)):
                    new_row.append(float(v))
                elif isinstance(v, str):
                    vv = v.strip()
                    if _NUM.fullmatch(vv):
                        new_row.append(float(vv))
                    else:
                        new_row.append(v)
                else:
                    return None

            norm.append(new_row)

        return norm

    except Exception as e:
        return None



def evaluate_table_list(output, gt):
    if output is None:
        return False
    
    pred_table = parse_table_list(output)
    
    # 维度不一致也没关系 
    
    pred_cells = table_to_cells(pred_table)
    gold_cells = table_to_cells(gt)

    
    n = len(gold_cells)
    m = len(pred_cells)
    
    
    if n == 0 and m == 0:
        return 1.0
    if n == 0 or m == 0:
        return 0.0

    # weight[i][j] = sim(gold_i, pred_j)
    weight = [[0.0 for _ in range(m)] for __ in range(n)]

    for i, (gr, gc, gv) in enumerate(gold_cells):
        gkey = f"{gr} | {gc}"
        for j, (pr, pc, pv) in enumerate(pred_cells):
            pkey = f"{pr} | {pc}"

            key_sim = esm_sim(gkey, pkey)

            ns = num_sim(pv, gv)
            weight[i][j] = key_sim * ns

    assignment = greedy_max(weight)

    matched_sum = 0.0
    for i in range(n):
        j = assignment[i]
        if j != -1 and weight[i][j] > 0:
            matched_sum += weight[i][j]

    precision = matched_sum / m
    recall = matched_sum / n
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall / (precision + recall))
    return f1
    
    


In [13]:
# 提取python代码
def extract_last_code_block(output):
    output_llm = exact_answer_llm(output, "code")

    block = re.search(
        r"(?is)answer:\s*```(?:python)?\s*\n(.*?)\n```",
        output_llm
    )

    if block:
        return block.group(1)

    return output

# 检测代码的有效性
def code_validity(output):
    if not isinstance(output, str):
        return False
    
    code = textwrap.dedent(output).strip()
    
    try:
        import matplotlib
        matplotlib.use("Agg")
    except Exception:
        pass
    
    try:
        exec(code, {})
        import matplotlib.pyplot as plt
        plt.close("all")
        
        return True
    except Exception:
        pass
    
    try:
        exec(code, {})
        import matplotlib.pyplot as plt
        plt.close("all")
        return True
    except Exception:
        return False
    
# 提取代码绘制的图表类型
# grouped_bar_chart, line_chart, bar_chart, barh_chart, pie_chart, scatter_chart, stacked_bar_chart, box_chart, radar, heatmap
def get_chart_type(code_str: str):
    """
    纯字符串提取图类型（不使用 ast）
    返回：
    grouped_bar_chart, line_chart, bar_chart, barh_chart,
    pie_chart, scatter_chart, stacked_bar_chart,
    box_chart, radar, heatmap
    """

    code = code_str.lower()

    # ----  radar 优先 ----
    if "projection='polar'" in code or 'projection="polar"' in code \
       or "polar=True" in code or "plt.polar(" in code:
        return "radar"

    # ---- heatmap ----
    if any(k in code for k in [
        "imshow(", "matshow(", "pcolormesh(", "pcolor("
    ]):
        return "heatmap"

    # ----  barh ----
    if "barh(" in code:
        return "barh_chart"

    # ----  bar ----
    if "bar(" in code:
        # stacked
        if "bottom=" in code:
            return "stacked_bar_chart"
        
        bar_calls = re.findall(r'bar\((.*?)\)', code)

        for call in bar_calls:
            x_part = call.split(",")[0].replace(" ", "")

            # grouped bar 常见偏移模式
            grouped_signals = [
                "+width", "-width",
                "+i*", "-i*",
                "+idx*", "-idx*",
                "+j*", "-j*",
                "+bar_width", "-bar_width",
                "+offset", "-offset",
                "width/2", "width*",
            ]

            if any(sig in x_part for sig in grouped_signals):
                return "grouped_bar_chart"
                
        return "bar_chart" 


    # ---- 5 其他类型 ----
    if "boxplot(" in code:
        return "box_chart"

    if "pie(" in code:
        return "pie_chart"

    if "scatter(" in code:
        return "scatter_chart"

    if "plot(" in code:
        return "line_chart"

    return None


# 用re提取出output中的所有[]数据， 返回一个二维list
def extract_data(output):
    pattern = r'\[[^\[\]]*\]'
    matches = re.findall(pattern, output)

    out = []
    for m in matches:
        cleaned = m.replace("null", "None")
        try:
            val = ast.literal_eval(cleaned)

            if (
                isinstance(val, list)
                and len(val) > 0
                and all(isinstance(x, (int, float)) for x in val)
            ):
                out.append(val)

        except Exception:
            continue

    return out if out else None

# 判断是否有对数据进行操作
import re

def extract_operation(output: str) -> str:
    code = output.lower()

    lines = code.split("\n")

    # 只分析赋值语句
    assign_lines = [line for line in lines if "=" in line and "==" not in line]

    joined = " ".join(assign_lines)

    # log
    if any(k in joined for k in ["log(", "np.log", "math.log"]):
        return "log"

    # mean
    if any(k in joined for k in [".mean(", "np.mean", "average("]):
        return "mean"

    # sort
    if any(k in joined for k in ["sorted(", ".sort(", "sort_values("]):
        return "ascending"

    # division
    if "/" in joined or "divide(" in joined:
        return "division"

    # subtraction
    if " - " in joined or "subtract(" in joined:
        return "subtraction"

    return "none"


In [14]:
# 下面是可读性判断
# 下面是可以运行代码提取的一些信息
import textwrap

def extract_readability(output: str):
    """
    运行代码并从 matplotlib figure 读取可读性信息
    不再包含 label 维度
    """

    code = textwrap.dedent(output).strip()

    try:
        import matplotlib
        matplotlib.use("Agg")
    except Exception:
        pass

    import matplotlib.pyplot as plt

    try:
        exec(code, {"plt": plt})
    except Exception:
        return None

    fig = plt.gcf()
    axes = fig.get_axes()

    if not axes:
        return None

    ax = axes[0]

    # ---------- 基础文本 ----------
    title = ax.get_title() or None
    xlabel = ax.get_xlabel() or None
    ylabel = ax.get_ylabel() or None
    legend = True if ax.get_legend() else None

    xticklabels = [t.get_text() for t in ax.get_xticklabels()]
    yticklabels = [t.get_text() for t in ax.get_yticklabels()]

    if all(x == "" for x in xticklabels):
        xticklabels = None
        
    if all(y == "" for y in yticklabels):
        yticklabels = None

    # ---------- style ----------
    style = {
        "color": None,
        "linestyle": None,
        "marker": None,
        "figsize": None,
        "fontsize": None
    }

    # figsize
    if fig.get_size_inches() is not None:
        style["figsize"] = True

    # 线图属性
    for line in ax.get_lines():
        if line.get_marker() not in (None, "None", ""):
            style["marker"] = True
        if line.get_linestyle() not in (None, "None", ""):
            style["linestyle"] = True
        if line.get_color() is not None:
            style["color"] = True

    # 柱状图颜色
    for container in ax.containers:
        for bar in container:
            if isinstance(bar, mpatches.Rectangle):
                if bar.get_facecolor() is not None:
                    style["color"] = True

    # fontsize（如果标题字体不同于默认值）
    try:
        import matplotlib
        if title and ax.title.get_fontsize() != matplotlib.rcParams["axes.titlesize"]:
            style["fontsize"] = True
    except Exception:
        pass

    return {
        "title": title,
        "xlabel": xlabel,
        "ylabel": ylabel,
        "legend": legend,
        "tick": {
            "xticklabels": xticklabels,
            "yticklabels": yticklabels
        },
        "style": style
    }

In [15]:
def rows_match_score(pred_2d, gt_2d, eps=1e-6):
    """
    按行匹配：
    - 允许 pred 行数 > gt
    - 每一行 gt 在 pred 中找最相似的一行
    - 返回 0~1 相似度
    """

    if not isinstance(pred_2d, list) or not isinstance(gt_2d, list):
        return 0.0

    if len(gt_2d) == 0:
        return 0.0

    def row_equal(r1, r2):
        if len(r1) != len(r2):
            return False
        for a, b in zip(r1, r2):
            if a is None and b is None:
                continue
            if isinstance(a, (int, float)) and isinstance(b, (int, float)):
                if abs(a - b) > eps:
                    return False
            else:
                if a != b:
                    return False
        return True

    matched = 0

    for gt_row in gt_2d:
        for pred_row in pred_2d:
            if row_equal(pred_row, gt_row):
                matched += 1
                break   # 找到一个就够了

    return matched / len(gt_2d)

In [16]:
# ========= 编辑距离相似度 =========
def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def levenshtein(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[m]

def edit_sim(a, b):
    # 同为 None：满分
    if a is None and b is None:
        return 1.0
    if a is None or b is None:
        return 0.0
    a2 = normalize_text(a)
    b2 = normalize_text(b)
    if len(a2) == 0 and len(b2) == 0:
        return 1.0
    dist = levenshtein(a2, b2)
    denom = max(len(a2), len(b2), 1)
    sim = 1.0 - dist / denom
    return max(0.0, min(1.0, sim))

In [17]:
# 测评代码 —— code
# 1. 给出的代码能不能运行
# 2. 使用的数据(行匹配) 图的类型  操作（升序 聚合等）
# 3. 可读性
def evaluate_code(output, gt):
    if output is None:
        return False
    
    score = 0

    output = extract_last_code_block(output)

    # 1. 判断代码能不能运行
    if(code_validity(output)):
        score = 0.2
    else:
        return 0

    # 2. 合法性 0.45
    # 2.1 图的类型是否正确 0.2
    gt_chart_type = gt.get("chart_type")
    pred_chart_type = get_chart_type(output)
    if pred_chart_type is not None and gt_chart_type is not None and pred_chart_type == gt_chart_type:
        score += 0.2
        
    else:
        return 0
    
    # 2.2 使用的数据是否正确 0.25
    gt_data = gt.get("data")  # 二维 list
    pred_data = extract_data(output)  # 可能抓到很多 list
    if gt_data and pred_data:
        sim = rows_match_score(pred_data, gt_data)
        if sim == 0:
            return 0
        score += 0.25 * sim
    
    # 2.3 对数据进行的操作是否 扣0.1
    gt_op = gt.get("operation")
    pred_op = extract_operation(output)
    if gt_op is not None:
        if pred_op != gt_op:
            score -= 0.1
    else:
        # gt 不要求 operation 时，你也可以选择不计分；这里保持不加分
        pass
    
    # 3. 可读性 0.35
    pred = extract_readability(output)
    if pred is None:
        return score
    
    # 3.1  三个文本字段：编辑距离相似度 ---  0.15
    s = 0.0
    s += 0.05 * edit_sim(gt.get("title"),  pred.get("title"))
    s += 0.05 * edit_sim(gt.get("xlabel"), pred.get("xlabel"))
    s += 0.05 * edit_sim(gt.get("ylabel"), pred.get("ylabel"))
    
    # 3.2 其他的 0.05 + 0.1 + 0.05
    # 取值
    gt_legend = gt.get("legend")
    pred_legend = pred.get("legend")

    # -------- legend --------
    if gt_legend == pred_legend:
        s += 0.05
        
    tick_items = [
        ("xticklabels", gt.get("tick", {}).get("xticklabels"), pred.get("tick", {}).get("xticklabels")),
        ("yticklabels", gt.get("tick", {}).get("yticklabels"), pred.get("tick", {}).get("yticklabels")),
    ]

    for _, gt_v, pred_v in tick_items:
        if gt_v == pred_v:
            s += 0.05
        elif gt_v is None and gt_legend is not None:
            s += 0.05
            
    # -------- style items --------
    style_items = [
        ("color", gt.get("style", {}).get("color"), pred.get("style", {}).get("color")),
        ("linestyle", gt.get("style", {}).get("linestyle"), pred.get("style", {}).get("linestyle")),
        ("marker", gt.get("style", {}).get("marker"), pred.get("style", {}).get("marker")),
        ("figsize", gt.get("style", {}).get("figsize"), pred.get("style", {}).get("figsize")),
        ("fontsize", gt.get("style", {}).get("fontsize"), pred.get("style", {}).get("fontsize")),
    ]

    for _, gt_v, pred_v in style_items:
        if gt_v == pred_v:
            s += 0.01
        elif gt_v is None and gt_legend is not None:
            s += 0.01
    
    score = score + s 

    return score

In [ ]:
DATA_PATHS = [
    "chart-table/cs/questions_text.json",
    "multicharts/cs/questions_text.json",
    "multitables/cs/questions_text.json"
]

def evaluate_dataset(subject_path, full_json_path):

    with open(full_json_path, "r", encoding="utf-8") as f:
        samples_data = json.load(f)

    stats = {
        "perceptual": {"score": 0, "count": 0},
        "reasoning": {"score": 0, "count": 0},
        "table to chart": {"score": 0, "count": 0},
        "chart to table": {"score": 0, "count": 0},
    }

    for sample in samples_data:
        score = 0

        print("-----sample-----")
        print(sample)

        # subject_path = 图片根目录
        output = call_llm(subject_path, sample)
        answer_type = sample["answer_type"]
        gt = sample["answer"]
        
        if answer_type == "numeric":
            score = evaluate_numeric(output, gt)
        elif answer_type == "numeric_list":
            score = evaluate_numeric_list(output, gt)
        elif answer_type == "text":
            score = evaluate_text(output, gt)
        elif answer_type == "choice":
            score = evaluate_choice(output, gt)
        elif answer_type == "choice_list":
            score = evaluate_choice_list(output, gt)
        elif answer_type == "table_list":
            score = evaluate_table_list(output, gt)
        elif answer_type == "code":
            score = evaluate_code(output, gt)

        question_type = sample["question_type"]

        if question_type in stats:
            stats[question_type]["count"] += 1
            stats[question_type]["score"] += score

    return stats


# ===== 全局统计 =====
global_stats = {
    "perceptual": {"score": 0, "count": 0},
    "reasoning": {"score": 0, "count": 0},
    "table to chart": {"score": 0, "count": 0},
    "chart to table": {"score": 0, "count": 0},
}

# ===== 主循环（只跑3个文件）=====
for full_json_path in DATA_PATHS:

    if not os.path.exists(full_json_path):
        print(f"不存在: {full_json_path}")
        continue

    print(f"\n====== Running {full_json_path} ======")

    # 正确：图片根目录就是 json 所在目录
    subject_path = os.path.dirname(full_json_path)

    local_stats = evaluate_dataset(subject_path, full_json_path)

    for k in local_stats:
        local_score = local_stats[k]["score"]
        local_count = local_stats[k]["count"]

        avg = local_score / local_count if local_count else 0
        print(f"{k}: Count={local_count}, Avg={avg:.4f}")

        global_stats[k]["score"] += local_score
        global_stats[k]["count"] += local_count


# ===== 全局平均 =====
print("\n====== Global Average ======")

for k in global_stats:
    total_score = global_stats[k]["score"]
    total_count = global_stats[k]["count"]

    avg = total_score / total_count if total_count else 0
    print(f"{k}: Total Count={total_count}, Global Avg={avg:.4f}")